In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


def initialize_plotting():
    import matplotlib as mpl
    %config InlineBackend.figure_format = 'svg'
    label_size = 20
    mpl.rcParams['xtick.labelsize'] = label_size
    mpl.rcParams['ytick.labelsize'] = label_size
    mpl.rcParams['legend.fontsize'] = 14
    plt.rc('font', family='serif')
    mpl.rcParams.update({'font.size': 16})
    mpl.rcParams['text.usetex'] = False
    mpl.rcParams['figure.dpi'] = 120
initialize_plotting()

In [ ]:
# Inciso b

# parámetros generales
N_senales = 3000
N_t = 1000
t_span = (0, 10)
t_eval = np.linspace(t_span[0], t_span[1], N_t)
sigma_ruido = 0.02

# Condiciones iniciales
y0 = [1.0, 0.0]

# distribuciones para los parámetros físicos
np.random.seed(42) # fijamso semilla 
gamma_vals = np.random.uniform(0.05, 1.0, N_senales)
k_vals = np.random.uniform(1.0, 5.0, N_senales)

# matrices para guardar los resultados
X_data = np.zeros((N_senales, N_t))
y_data = np.column_stack((gamma_vals, k_vals))

# definimos  sistema de edos
def oscilador_amortiguado(t, y, gamma, k):
    x, v = y
    dxdt = v
    dvdt = -gamma * v - k * x
    return [dxdt, dvdt]

# generamos las señales

for i in range(N_senales):
    
    sol = solve_ivp(oscilador_amortiguado, t_span, y0, t_eval=t_eval, 
                    args=(gamma_vals[i], k_vals[i]), method='RK45')
    x_t = sol.y[0]
    # ruido gaussiano
    ruido = np.random.normal(0, sigma_ruido, N_t)
    x_obs = x_t + ruido
    X_data[i] = x_obs

# ejemplos para discusión
plt.figure(figsize=(12, 6))

# elegimos 3 índices al azar para graficar
indices_plot = np.random.choice(N_senales, 3, replace=False)

for idx in indices_plot:
    g = gamma_vals[idx]
    k = k_vals[idx]
    plt.plot(t_eval, X_data[idx], label=f'$\gamma={g:.2f}, k={k:.2f}$')

plt.xlabel('Tiempo (t)')
plt.ylabel('Posición observada $x_{obs}(t)$')
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
#inciso c
# separamos datos (80% train, 20% test)

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.20, random_state=42
)

# Random Forest

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf_model.predict(X_test)

# Cálculo de RMSE (columna 0 gamma, columna 1 k)
rmse_gamma_rf = np.sqrt(mean_squared_error(y_test[:, 0], y_pred_rf[:, 0]))
rmse_k_rf = np.sqrt(mean_squared_error(y_test[:, 1], y_pred_rf[:, 1]))

print(f"--- Desempeño Random Forest ---")
print(f"RMSE Gamma: {rmse_gamma_rf:.4f}")
print(f"RMSE k:     {rmse_k_rf:.4f}\n")


# 3. Modelo 2: Red Neuronal simple con PyTorch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# entrenamiento por lotes (batches)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Definir la red neuronal
class MLPRegressor(nn.Module):
    def __init__(self, input_size):
        super(MLPRegressor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # 2 salidas para [gamma, k]
        )
        
    def forward(self, x):
        return self.network(x)


input_dim = X_train.shape[1]
mlp_model = MLPRegressor(input_size=input_dim)

# Función de pérdida y optimizador 
criterion = nn.MSELoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# Ciclo de entrenamiento 
epochs = 50

for epoch in range(epochs):
    mlp_model.train() # Modo entrenamiento
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        optimizer.zero_grad() # Limpiar gradientes
        outputs = mlp_model(inputs) # Forward pass
        loss = criterion(outputs, targets) # Calcular error
        loss.backward() # Backpropagation
        optimizer.step() # Actualizar pesos
        
        running_loss += loss.item()
        
    # Imprimir cada 10 épocas
    if (epoch + 1) % 10 == 0:
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}], Loss (MSE): {avg_loss:.4f}")

# Evaluación de la Red Neuronal
mlp_model.eval() # Modo evaluación (desactiva dropout/batchnorm si los hubiera)
with torch.no_grad(): # No necesitamos calcular gradientes para testear
    y_pred_mlp_tensor = mlp_model(X_test_tensor)
    y_pred_mlp = y_pred_mlp_tensor.numpy()

# Cálculo de RMSE para MLP
rmse_gamma_mlp = np.sqrt(mean_squared_error(y_test[:, 0], y_pred_mlp[:, 0]))
rmse_k_mlp = np.sqrt(mean_squared_error(y_test[:, 1], y_pred_mlp[:, 1]))

print(f"\n--- Desempeño MLP PyTorch ---")
print(f"RMSE Gamma: {rmse_gamma_mlp:.4f}")
print(f"RMSE k:     {rmse_k_mlp:.4f}")

In [ ]:
# Inciso d

sigmas = [0, 0.01, 0.02, 0.05, 0.10]

rmse_gamma_train, rmse_gamma_test = [], []
rmse_k_train, rmse_k_test = [], []

# Usamos RandomForest para agilizar el barrido
rf_model_noise = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)

for sigma in sigmas:
    X_ruidoso = np.zeros((N_senales, N_t))
    
    for i in range(N_senales):
        sol = solve_ivp(oscilador_amortiguado, t_span, y0, t_eval=t_eval, 
                        args=(gamma_vals[i], k_vals[i]), method='RK45')
        ruido = np.random.normal(0, sigma, N_t)
        X_ruidoso[i] = sol.y[0] + ruido
        
    X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
        X_ruidoso, y_data, test_size=0.20, random_state=42
    )
    
    # entrenamiento
    rf_model_noise.fit(X_train_n, y_train_n)
    
    # Predicciones
    y_pred_train = rf_model_noise.predict(X_train_n)
    y_pred_test = rf_model_noise.predict(X_test_n)
    
    # Cálculo de RMSE para Train y Test
    rmse_gamma_train.append(np.sqrt(mean_squared_error(y_train_n[:, 0], y_pred_train[:, 0])))
    rmse_k_train.append(np.sqrt(mean_squared_error(y_train_n[:, 1], y_pred_train[:, 1])))
    
    rmse_gamma_test.append(np.sqrt(mean_squared_error(y_test_n[:, 0], y_pred_test[:, 0])))
    rmse_k_test.append(np.sqrt(mean_squared_error(y_test_n[:, 1], y_pred_test[:, 1])))

# graficamos resultados
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(sigmas, rmse_gamma_train, 'o--', label='Train', color='purple')
ax1.plot(sigmas, rmse_gamma_test, 's-', label='Test', color='green')
ax1.set_xlabel('$\sigma$ (Ruido)')
ax1.set_ylabel('$RMSE_{\gamma}$')
ax1.legend()
ax1.grid(True, alpha=0.5)

ax2.plot(sigmas, rmse_k_train, 'o--', label='Train', color='purple')
ax2.plot(sigmas, rmse_k_test, 's-', label='Test', color='green')
ax2.set_xlabel('$\sigma$ (Ruido)')
ax2.set_ylabel('$RMSE_{k}$')
ax2.legend()
ax2.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()